In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import base64
import os
toilets_data = {
    "文華樓": {
        "scores": [4.67, 3.50, 4.67, 2.83], "x": 6.5, "y": 5.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "78 / 95",
        "review": "有開窗通風且採光佳且有特別打燈，乾淨且提供肥皂和洗手乳。"
    },
    "文友樓": {
        "scores": [4.67, 5.00, 4.00, 2.83], "x": 7.0, "y": 5.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "82 / 95",
        "review": "正常且有屏風避免直接面對走廊，採光佳也有加開窗戶增加通風。"
    },
    "文開樓": {
        "scores": [3.17, 3.50, 3.00, 3.00], "x": 8.5, "y": 5.0,
        "stars": "⭐⭐⭐⭐", "total": "65 / 95",
        "review": "有一點點異味和積水和積水，中規中矩的普通廁所。"
    },
    "公博樓": {
        "scores": [4.00, 3.50, 5.00, 2.33], "x": 8.0, "y": 2.0,
        "stars": "⭐⭐⭐⭐", "total": "72 / 95",
        "review": "通風良好光線充足隔間也寬敞，缺乏設計的一般普通廁所。"
    },
    "藝術學院": {
        "scores": [4.83, 3.50, 5.00, 3.00], "x": 9.0, "y": 1.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "81 / 95",
        "review": "有香味且很適合拍照，植栽和畫作裝飾像是百貨公司廁所。"
    },
    "外語學院": {
        "scores": [3.50, 3.50, 2.67, 2.50], "x": 3.5, "y": 3.5,
        "stars": "⭐⭐⭐⭐", "total": "63 / 95",
        "review": "通風普通採光稍差有點昏暗，隔間稍小但足夠，一間有設計過的普通廁所。"
    },
    "聖言樓": {
        "scores": [2.00, 2.50, 2.67, 2.17], "x": 3.5, "y": 7.5,
        "stars": "⭐⭐⭐", "total": "48 / 95",
        "review": "馬桶很髒且鏡子老舊鏽蝕，採光差很昏暗且什麼都沒提供，簡直是恐怖片廁所。"
    },
    "淨心堂": {
        "scores": [5.00, 5.00, 5.00, 3.67], "x": 4.5, "y": 6.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "92 / 95",
        "review": "整體設計好看，空間乾淨寬敞，像飯店廁所，會讓人想待在裡面。"
    }
}
image_filename = 'middle.png'
encoded_image = ""
if os.path.exists(image_filename):
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode('utf-8')
app = dash.Dash(__name__)
MAP_WIDTH = 650
MAP_HEIGHT = 600
RADAR_WIDTH = 500
RADAR_HEIGHT = 440
app.layout = html.Div(style={'backgroundColor': '#1E1E1E', 'padding': '20px', 'display': 'flex', 'fontFamily': 'Microsoft JhengHei'}, children=[
    html.Div(style={'width': '55%'}, children=[
        dcc.Graph(id='fujen-map-mid', config={'displayModeBar': False})
    ]),
    html.Div(style={'width': '45%', 'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}, children=[
        dcc.Graph(id='radar-chart-mid', config={'displayModeBar': False}),
        html.Div(id='review-box-mid', style={
            'width': '90%', 'marginTop': '10px', 'padding': '15px',
            'backgroundColor': 'rgba(255, 255, 255, 0.05)', 'borderRadius': '8px',
            'border': '1px solid rgba(0, 255, 204, 0.3)', 'color': '#FFFFFF'
        })
    ])
])
def draw_base_map():
    fig = go.Figure()
    for name, info in toilets_data.items():
        fig.add_trace(go.Scatter(
            x=[info["x"]], y=[info["y"]],
            mode="markers+text",
            marker=dict(size=18, color='#00FFCC', line=dict(color='#FFFFFF', width=1.5)),
            text=[name],
            textposition="top center",
            textfont=dict(size=13, color="#FFFFFF", family="Microsoft JhengHei"),
            customdata=[[info["x"], info["y"]]],
            hoverinfo="text",
            hovertext=f"<b>🏢 {name}</b><br>🔍 游標移入，下方即刻呈現雷達圖與總評",
            hovertemplate="%{hovertext}<extra></extra>",
            showlegend=False
        ))
    img_source = f"data:image/png;base64,{encoded_image}" if encoded_image else ""
    fig.update_layout(
        images=[dict(
            source=img_source, xref="x", yref="y", x=0, y=10, sizex=10, sizey=10,
            sizing="stretch", opacity=0.9, layer="below"
        )],
        title=dict(text="<b>🗺️ 輔大中間區域 — 廁所空間生態地圖</b>", x=0.5, font=dict(size=20, color='#FFFFFF')),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 10.5], fixedrange=True),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 10.5], fixedrange=True),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        width=MAP_WIDTH, height=MAP_HEIGHT, margin=dict(l=20, r=20, t=60, b=20)
    )
    return fig
def draw_radar(building_name):
    scores = toilets_data[building_name]["scores"]
    s1, s2, s3, s4 = scores
    x_coords = [0, s2, 0, -s4, 0]
    y_coords = [s1, 0, -s3, 0, s1]
    hover_texts = [
        f"1.環境衛生: {s1:.2f}分", f"2.設施完備: {s2:.2f}分",
        f"3.空間舒適: {s3:.2f}分", f"4.特殊機能: {s4:.2f}分", f"1.環境衛生: {s1:.2f}分"
    ]
    fig = go.Figure()
    for r in range(1, 6):
        fig.add_trace(go.Scatter(
            x=[0, r, 0, -r, 0], y=[r, 0, -r, 0, r],
            mode='lines', line=dict(color='rgba(255,255,255,0.18)', width=1),
            showlegend=False, hoverinfo='skip'
        ))
        fig.add_trace(go.Scatter(
            x=[0.15], y=[r-0.15], mode='text', text=[str(r)],
            textfont=dict(color='rgba(255,255,255,0.4)', size=10),
            showlegend=False, hoverinfo='skip'
        ))
    fig.add_trace(go.Scatter(
        x=[-5, 5, None, 0, 0], y=[0, 0, None, -5, 5],
        mode='lines', line=dict(color='rgba(255,255,255,0.2)', width=1.5),
        showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=x_coords, y=y_coords,
        fill='toself', fillcolor='rgba(0, 200, 150, 0.25)',
        line=dict(color='#00FFCC', width=4), marker=dict(color='#00FFCC', size=9),
        hoverinfo='text', text=hover_texts, hovertemplate="%{text}<extra></extra>"
    ))
    labels = [
        dict(x=0, y=5.5, text="1.環境衛生"), dict(x=5.9, y=0, text="2.設施完備"),
        dict(x=0, y=-5.5, text="3.空間舒適"), dict(x=-5.9, y=0, text="4.特殊機能")
    ]
    for label in labels:
        fig.add_trace(go.Scatter(
            x=[label['x']], y=[label['y']], mode='text', text=[f"<b>{label['text']}</b>"],
            textfont=dict(size=14, color='#00FFCC', family='Microsoft JhengHei'),
            showlegend=False, hoverinfo='skip'
        ))
    fig.update_layout(
        title=dict(text=f"<b>{building_name} 綜合評鑑鑽石圖</b>", x=0.5, font=dict(size=18, color='#FFFFFF')),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-6.8, 6.8], fixedrange=True),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-6.8, 6.8], fixedrange=True),
        showlegend=False, paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        width=RADAR_WIDTH, height=RADAR_HEIGHT, margin=dict(l=20, r=20, t=50, b=10)
    )
    return fig
@app.callback(
    [Output('radar-chart-mid', 'figure'),
     Output('review-box-mid', 'children')],
    Input('fujen-map-mid', 'hoverData')
)
def update_dashboard_on_hover(hoverData):
    building_name = "文華樓"
    if hoverData and 'points' in hoverData and len(hoverData['points']) > 0:
        hover_x = hoverData['points'][0].get('x', None)
        hover_y = hoverData['points'][0].get('y', None)
        for b_name, b_info in toilets_data.items():
            if b_info["x"] == hover_x and b_info["y"] == hover_y:
                building_name = b_name
                break
    info = toilets_data[building_name]
    review_layout = html.Div([
        html.Div([
            html.Span(f"🏢 {building_name} ", style={'fontSize': '18px', 'fontWeight': 'bold', 'color': '#00FFCC'}),
            html.Span(f" {info['stars']}", style={'fontSize': '16px', 'marginLeft': '10px'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}),
        html.Div([
            html.B("📊 綜合實測總分：", style={'color': '#FF9900'}),
            html.Span(info['total'], style={'fontSize': '16px', 'fontWeight': 'bold'})
        ], style={'marginBottom': '8px'}),
        html.Div([
            html.B("📝 實地考察總評：", style={'color': '#00FFFF'}),
            html.P(info['review'], style={'fontSize': '13px', 'color': '#DDDDDD', 'lineHeight': '1.5', 'margin': '0'})
        ])
    ])
    return draw_radar(building_name), review_layout
@app.callback(Output('fujen-map-mid', 'figure'), Input('fujen-map-mid', 'id'))
def init_map(_): return draw_base_map()
# 🛠️ 加在這裡：強制把最新做好的地圖導覽存成網頁檔
import plotly.io as pio
pio.write_html(draw_base_map(), file="mmap.html", auto_open=False)
if __name__ == '__main__':
    import plotly.io as pio
    pio.write_html(draw_radar(list(toilets_data.keys())[0]), file="mid.html", include_plotlyjs="cdn")
    app.run(jupyter_mode='inline', port=8060)

In [2]:
from pyngrok import ngrok

# 🛠️ 直奔主題，直接開啟通道對接 8060
try:
    public_url = ngrok.connect(8060)
    print("\n🎉🎉🎉 終於成功拿到外部互動連結了！！！ 🎉🎉🎉")
    print(f"請複製這串網址發給組員或貼進 Canva：\n\n{public_url}\n")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print("⚠️ 提醒：報告完畢前，請維持地圖專案執行，網頁才能正常互動喔！")
except Exception as e:
    print(f"發生錯誤：{e}")


🎉🎉🎉 終於成功拿到外部互動連結了！！！ 🎉🎉🎉
請複製這串網址發給組員或貼進 Canva：

NgrokTunnel: "https://cofounder-uprising-spiritism.ngrok-free.dev" -> "http://localhost:8060"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ 提醒：報告完畢前，請維持地圖專案執行，網頁才能正常互動喔！
